# LACE — paper results notebook

Every table and figure in the paper is derived here from the run JSONs in
`research/lace/runs/` (written by `adajepa` CLI commands and
the deployment replay harness). Nothing is hand-entered.

**Arms**: `frozen` (no TTA) · `unlaced` (AdaJEPA: self-referential target) ·
`laced-frozen` (LACE: frozen anchor target + anchored goal) · `laced-ema`
(LACE: slow-EMA anchor).

Gates are pre-registered in `../GATES.md`; each section below reports its
gate verdict.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RUNS = Path("..") / "runs"

SEEN = ["shape:T", "shape:L", "shape:Z", "shape:plus"]
UNSEEN = ["shape:I", "shape:smallT", "shape:cube"]


def load(name):
    p = RUNS / name
    return json.loads(p.read_text()) if p.exists() else None


def pooled_probe_auc(data, shifts, arm):
    """Pool probe success logits/labels across shifts for a stable AUC."""
    from math import isfinite
    logits, labels = [], []
    for s in shifts:
        cell = data["shifts"].get(s, {}).get(arm)
        if not cell:
            continue
        for ep in cell["episodes"]:
            logits += ep["probe_succ_logit"]
            labels += ep["label_succ"]
    logits, labels = np.array(logits), np.array(labels)
    pos, neg = logits[labels > 0.5], logits[labels <= 0.5]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan"), len(logits)
    wins = (pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()
    return float(wins / (len(pos) * len(neg))), len(logits)


def pooled_progress_corr(data, shifts, arm):
    prog, dist = [], []
    for s in shifts:
        cell = data["shifts"].get(s, {}).get(arm)
        if not cell:
            continue
        for ep in cell["episodes"]:
            prog += ep["probe_prog"]
            dist += ep["label_dist"]
    if len(prog) < 3:
        return float("nan")
    return float(np.corrcoef(np.array(prog), -np.array(dist))[0, 1])


def mean_success(data, shifts, arm):
    vals = [data["shifts"][s][arm]["success_rate"] for s in shifts if s in data["shifts"]]
    return float(np.mean(vals)) if vals else float("nan")


def mean_drift(data, shifts, arm):
    vals = [
        data["shifts"][s][arm].get("goal_drift_final_mean", np.nan)
        for s in shifts if s in data["shifts"]
    ]
    return float(np.nanmean(vals)) if vals else float("nan")

## G0 — benchmark validity (PushObj-mini)

Gate: seen-vs-unseen frozen success gap ≥ 10 pp; `unlaced` improves unseen
success; tasks oracle-solvable (verified separately, 8/8).

In [2]:
e2p = load("e2_pushobj_cem.json")
if e2p:
    arms = list(next(iter(e2p["shifts"].values())).keys())
    rows = []
    for arm in arms:
        auc_seen, n_seen = pooled_probe_auc(e2p, SEEN, arm)
        auc_unseen, n_unseen = pooled_probe_auc(e2p, UNSEEN, arm)
        rows.append({
            "arm": arm,
            "seen success %": round(mean_success(e2p, SEEN, arm), 1),
            "unseen success %": round(mean_success(e2p, UNSEEN, arm), 1),
            "probe succAUC (seen)": round(auc_seen, 3),
            "probe succAUC (unseen)": round(auc_unseen, 3),
            "progress corr": round(pooled_progress_corr(e2p, SEEN + UNSEEN, arm), 3),
            "goal drift": round(mean_drift(e2p, SEEN + UNSEEN, arm), 3),
        })
    df = pd.DataFrame(rows).set_index("arm")
    display(df)
    gap = df.loc["frozen", "seen success %"] - df.loc["frozen", "unseen success %"]
    g0_1 = gap >= 10
    g0_2 = df.loc["unlaced", "unseen success %"] > df.loc["frozen", "unseen success %"]
    print(f"G0.1 seen-unseen gap = {gap:.1f} pp -> {'PASS' if g0_1 else 'FAIL'}")
    print(f"G0.2 unlaced improves unseen -> {'PASS' if g0_2 else 'FAIL'}")
else:
    print("e2_pushobj_cem.json not present yet")

,seen success %,unseen success %,probe succAUC (seen),probe succAUC (unseen),progress corr,goal drift
arm,,,,,,
frozen,23.3,21.1,0.878,0.901,0.618,0.000
unlaced,20.8,22.2,0.868,0.855,0.599,0.111
laced-frozen,19.2,20.0,0.878,0.712,0.611,0.000
laced-ema,24.2,16.7,0.861,0.768,0.576,0.008


G0.1 seen-unseen gap = 2.2 pp -> FAIL
G0.2 unlaced improves unseen -> PASS


## E1 — characterization grid (maze, recipes × target source)

Gate G1: `unlaced` at full recipe must damage frozen consumers (probe AUC or
goal-latent drift); `laced-frozen` at the same recipe keeps ≥ 80 % of the
divergence reduction with probe AUC within 0.02 of frozen.

In [3]:
for fname, title in [("e1_maze_high_damping.json", "asymmetric LR (paper default)"),
                     ("e1_maze_high_damping_symlr.json", "SYMMETRIC encoder LR 3e-4")]:
    d = load(fname)
    if not d:
        print(fname, "missing"); continue
    rows = []
    for name, cell in d["arms"].items():
        p = cell.get("probes") or {}
        rows.append({
            "arm": name,
            "success %": cell["success_rate"],
            "± std": cell["success_std"],
            "probe succAUC": p.get("success_auc"),
            "probe progCorr": p.get("progress_corr"),
            "probe stateErr": p.get("state_err"),
            "goal drift": cell.get("goal_drift_final_mean"),
        })
    print(f"\n=== {title} ===")
    display(pd.DataFrame(rows).set_index("arm"))


=== asymmetric LR (paper default) ===


,success %,± std,probe succAUC,probe progCorr,probe stateErr,goal drift
arm,,,,,,
frozen,56.67,10.00,0.9647,0.9488,0.1736,None
student:predlast+enclast@x1,83.33,3.33,0.9760,0.9668,0.1780,None
student:predlast+enclast@x0.2,76.67,3.33,0.9666,0.9142,0.1738,None
student:predlast@x1,80.00,6.67,0.9738,0.8655,0.1704,None
student:predlast@x0.2,63.33,3.33,0.9619,0.9074,0.1675,None
frozen:predlast+enclast@x1,83.33,3.33,0.9782,0.9680,0.1726,None
frozen:predlast+enclast@x0.2,76.67,3.33,0.9619,0.9491,0.1722,None
frozen:predlast@x1,80.00,6.67,0.9738,0.8655,0.1704,None
frozen:predlast@x0.2,63.33,3.33,0.9619,0.9074,0.1675,None



=== SYMMETRIC encoder LR 3e-4 ===


,success %,± std,probe succAUC,probe progCorr,probe stateErr,goal drift
arm,,,,,,
frozen,56.67,10.00,0.9647,0.9488,0.1736,None
student:predlast+enclast@x1,76.67,3.33,0.9773,0.9044,0.3026,None
student:predlast+enclast@x0.2,70.00,16.67,0.9716,0.9125,0.2101,None
frozen:predlast+enclast@x1,76.67,10.00,0.9793,0.9130,0.2357,None
frozen:predlast+enclast@x0.2,60.00,6.67,0.9797,0.9066,0.2091,None


## E2 — full shift suites (PushObj shapes + maze) & E4 goal drift

Gate G2: `laced-frozen` success ≥ `unlaced` − 5 pp on every shift family; no
in-distribution harm vs frozen (−3 pp). E4: goal drift is zero for anchored
arms by construction — the table shows what unlaced does instead.

In [4]:
def suite_table(data, groups):
    arms = list(next(iter(data["shifts"].values())).keys())
    rows = []
    for arm in arms:
        row = {"arm": arm}
        for gname, shifts in groups:
            row[f"{gname} succ %"] = round(mean_success(data, shifts, arm), 1)
            auc, _ = pooled_probe_auc(data, shifts, arm)
            row[f"{gname} AUC"] = round(auc, 3)
        row["goal drift"] = round(
            mean_drift(data, [s for _, ss in groups for s in ss], arm), 3)
        rows.append(row)
    return pd.DataFrame(rows).set_index("arm")


for fname, groups, title in [
    ("e2_pushobj_cem.json", [("seen", SEEN), ("unseen", UNSEEN)], "PushObj (CEM)"),
    ("e2_pushobj_gd.json", [("subset", ["shape:T", "shape:I", "shape:cube"])], "PushObj (GD)"),
    ("e2_maze_cem.json",
     [("in-dist", ["default"]),
      ("dynamics", ["low_mass", "high_damping"]),
      ("visual", ["blur", "snp", "dark", "red_agent"])], "PointMaze (CEM)"),
    ("e2_maze_layout_cem.json",
     [("in-dist", ["layout:0"]),
      ("held-out layouts", ["layout:100", "layout:101", "layout:102"])],
     "PointMaze layouts (CEM, diverse model)"),
    ("e2_maze_gd.json",
     [("dynamics", ["low_mass", "high_damping"])], "PointMaze (GD)"),
]:
    d = load(fname)
    if not d:
        print(fname, "missing"); continue
    print(f"\n=== {title} ===")
    display(suite_table(d, groups))


=== PushObj (CEM) ===


,seen succ %,seen AUC,unseen succ %,unseen AUC,goal drift
arm,,,,,
frozen,23.3,0.878,21.1,0.901,0.000
unlaced,20.8,0.868,22.2,0.855,0.111
laced-frozen,19.2,0.878,20.0,0.712,0.000
laced-ema,24.2,0.861,16.7,0.768,0.008



=== PushObj (GD) ===


,subset succ %,subset AUC,goal drift
arm,,,
frozen,20.0,0.915,0.000
unlaced,20.0,0.945,0.102
laced-frozen,25.6,0.887,0.000



=== PointMaze (CEM) ===


,in-dist succ %,in-dist AUC,dynamics succ %,dynamics AUC,visual succ %,visual AUC,goal drift
arm,,,,,,,
frozen,80.0,0.995,78.3,0.948,66.7,0.819,0.000
unlaced,100.0,0.981,90.0,0.957,71.7,0.809,0.205
laced-frozen,93.3,0.986,88.3,0.960,72.5,0.837,0.000
laced-ema,96.7,0.984,83.3,0.964,71.7,0.841,0.023



=== PointMaze layouts (CEM, diverse model) ===


,in-dist succ %,in-dist AUC,held-out layouts succ %,held-out layouts AUC,goal drift
arm,,,,,
frozen,66.7,0.887,35.6,0.850,0.000
unlaced,70.0,0.883,45.6,0.808,0.164
laced-frozen,73.3,0.850,38.9,0.831,0.000
laced-ema,73.3,0.855,44.4,0.835,0.014



=== PointMaze (GD) ===


,dynamics succ %,dynamics AUC,goal drift
arm,,,
frozen,46.7,0.985,0.000
unlaced,40.0,0.994,0.073
laced-frozen,48.3,0.993,0.000


## E3 / E5 / E6 — ablations

E3 (gate G3): symmetric-LR sweep — does the student target degrade where the
anchored target stays safe? E5: shape-diversity low-data grid. E6: EMA-decay
sweep for the anchor.

In [5]:
e3 = load("e3_lr_sweep.json")
if e3:
    rows = []
    for source, cells in e3["cells"].items():
        for lr, cell in zip(e3["enc_lrs"], cells):
            rows.append({
                "source": source, "enc_lr": lr,
                "success %": cell["success_rate"],
                "probe succAUC": (cell.get("probes") or {}).get("success_auc"),
            })
    display(pd.DataFrame(rows).pivot(index="enc_lr", columns="source",
                                     values=["success %", "probe succAUC"]))

e5 = load("e5_lowdata.json")
if e5:
    rows = []
    for arm, cells in e5["cells"].items():
        for k, cell in zip(e5["budgets"], cells):
            rows.append({"arm": arm, "K shapes": k,
                         "unseen success %": cell["success_rate"],
                         "probe succAUC": cell["probes"]["success_auc"]})
    display(pd.DataFrame(rows).pivot(index="K shapes", columns="arm",
                                     values=["unseen success %", "probe succAUC"]))

e6 = load("e6_ema_decay.json")
if e6:
    display(pd.DataFrame([
        {"tau": t, "unseen success %": c["success_rate"],
         "probe succAUC": c["probes"]["success_auc"]}
        for t, c in zip(e6["decays"], e6["cells"])
    ]).set_index("tau"))

success %         probe succAUC        
source     frozen student        frozen student
enc_lr                                         
0.00001     73.33   66.67        0.9695  0.9684
0.00010     73.33   86.67        0.9731  0.9705
0.00030     66.67   80.00        0.9782  0.9803
0.00100     86.67   33.33        0.9802  0.9909

unseen success %                      probe succAUC               \
arm                frozen laced-frozen unlaced        frozen laced-frozen   
K shapes                                                                    
1                    20.0         15.0    20.0        0.6746       0.8984   
2                    15.0         20.0    25.0        0.8426       0.9235   
4                    10.0         10.0    10.0        0.9601       0.9072   

                  
arm      unlaced  
K shapes          
1         0.9504  
2         0.9733  
4         0.9549

,unseen success %,probe succAUC
tau,,
0.900,5.0,0.6763
0.996,15.0,0.8510
1.000,10.0,0.9072


## E7 — deployed screen-agent replay (2×2 grid)

Prequential replay of recorded deployment runs: each step is scored before
adapting on it. Gate: `laced-frozen` at FULL recipe must keep the divergence
gain with success-AUC ≈ frozen (where `unlaced` full dropped 0.947 → 0.846).
`*_enc` metrics score the deployed heads on the encoded true next frame —
isolating encoder-space damage (what LACE anchors) from predictor-output
drift.

In [6]:
rows = []
for arm, recipe, fname in [
    ("unlaced", "full", "e7_swm_unlaced_full.json"),
    ("unlaced", "low", "e7_swm_unlaced_low.json"),
    ("laced-frozen", "full", "e7_swm_laced_full.json"),
    ("laced-frozen", "low", "e7_swm_laced_low.json"),
]:
    d = load(fname)
    if not d:
        print(fname, "missing"); continue
    s = d["second_half"]
    rows.append({
        "arm": arm, "recipe": recipe,
        "2nd-half div reduction %": round(s["div_reduction_pct"], 1),
        "cf_top1 frozen→adapted": f"{s['frozen_top1']:.3f}→{s['adapted_top1']:.3f}",
        "succAUC (pred)": round(d.get("adapted_success_auc", float("nan")), 3),
        "succAUC (enc)": round(d.get("adapted_success_auc_enc", float("nan")), 3),
        "progAUC (enc)": round(d.get("adapted_progress_auc_enc", float("nan")), 3),
    })
if rows:
    d0 = load("e7_swm_unlaced_full.json") or load("e7_swm_laced_full.json")
    print(f"frozen heads: succAUC(pred)={d0.get('frozen_success_auc'):.3f} "
          f"succAUC(enc)={d0.get('frozen_success_auc_enc', float('nan')):.3f}")
    display(pd.DataFrame(rows).set_index(["arm", "recipe"]))

frozen heads: succAUC(pred)=0.933 succAUC(enc)=0.695


2nd-half div reduction % cf_top1 frozen→adapted  \
arm          recipe                                                    
unlaced      full                        44.6            0.612→0.535   
             low                         31.9            0.612→0.593   
laced-frozen full                        43.6            0.612→0.534   
             low                         31.9            0.612→0.593   

                     succAUC (pred)  succAUC (enc)  progAUC (enc)  
arm          recipe                                                
unlaced      full             0.826          0.693          0.693  
             low              0.907          0.695          0.714  
laced-frozen full             0.838          0.696          0.702  
             low              0.907          0.695          0.714

In [7]:
# Regenerate every paper figure from the JSONs.
%run ../scripts/paper_figs.py

[e1]


  wrote paper/figures/e1_maze_grid.pdf
  wrote paper/figures/e1_maze_symlr.pdf
[e2]


  wrote paper/figures/e2_pushobj_shapes.pdf


  wrote paper/figures/e2_pushobj_curves.pdf
  wrote paper/figures/e2_maze_shifts.pdf


  wrote paper/figures/e2_maze_curves.pdf


  wrote paper/figures/e2_maze_layouts.pdf
[e3]


  wrote paper/figures/e3_lr_sweep.pdf
[e4]


  wrote paper/figures/e4_goal_drift.pdf
[e5]


  wrote paper/figures/e5_lowdata.pdf
[e6]
  wrote paper/figures/e6_ema_decay.pdf
[e7]
  wrote paper/figures/e7_swm_replay.pdf
[e7b]


  wrote paper/figures/e7b_mechanism.pdf
